<a href="https://colab.research.google.com/github/Mebay821/health-hackathon-project/blob/main/diabetes_risk_screening_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1
!pip -q install "gradio>=5,<6" reportlab pytesseract requests gTTS
print("Installed.")

Installed.


In [17]:

 import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","DiabetesPedigree","Age","Outcome"]
df = pd.read_csv(URL, names=cols)

print(df.shape)
print(df["Outcome"].mean())
df.head()



(768, 9)
0.3489583333333333


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigree,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [18]:
X = df[["Age","BMI","BloodPressure","Glucose"]]
y = df["Outcome"]

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)

print(len(X_train), len(X_val), len(X_test))

460 154 154


In [19]:
def report(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"{name}: accuracy {accuracy_score(y_true,y_pred):.1%}, recall {recall_score(y_true,y_pred):.1%}, "
          f"missed cases {fn}, false alarms {fp}")

report("Baseline", y_val, np.zeros(len(y_val)))

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)
report("Model", y_val, model.predict(X_val))

Baseline: accuracy 64.9%, recall 0.0%, missed cases 54, false alarms 0
Model: accuracy 73.4%, recall 74.1%, missed cases 14, false alarms 27


In [20]:
probs = model.predict_proba(X_val)[:,1]
for t in [0.5, 0.4, 0.3, 0.2]:
    report(f"threshold {t}", y_val, (probs >= t).astype(int))

threshold 0.5: accuracy 73.4%, recall 74.1%, missed cases 14, false alarms 27
threshold 0.4: accuracy 71.4%, recall 85.2%, missed cases 8, false alarms 36
threshold 0.3: accuracy 66.2%, recall 88.9%, missed cases 6, false alarms 46
threshold 0.2: accuracy 57.8%, recall 98.1%, missed cases 1, false alarms 64


In [21]:
t = 0.3
test_probs = model.predict_proba(X_test)[:,1]
report("Baseline (test)", y_test, np.zeros(len(y_test)))
report(f"Model, threshold {t} (test)", y_test, (test_probs >= t).astype(int))

Baseline (test): accuracy 64.9%, recall 0.0%, missed cases 54, false alarms 0
Model, threshold 0.3 (test): accuracy 68.8%, recall 92.6%, missed cases 4, false alarms 44


In [22]:
test_pred = (test_probs >= 0.3).astype(int)
groups = pd.cut(X_test["Age"], bins=[0, 30, 45, 100], labels=["under 30", "30-45", "over 45"])

for g in ["under 30", "30-45", "over 45"]:
    mask = (groups == g).values
    n_sick = int(y_test[mask].sum())
    tn, fp, fn, tp = confusion_matrix(y_test[mask], test_pred[mask]).ravel()
    print(f"{g}: {mask.sum()} patients, {n_sick} sick, caught {tp}, missed {fn}, false alarms {fp}")

under 30: 89 patients, 18 sick, caught 16, missed 2, false alarms 26
30-45: 40 patients, 26 sick, caught 24, missed 2, false alarms 9
over 45: 25 patients, 10 sick, caught 10, missed 0, false alarms 9


In [23]:
val_groups = pd.cut(X_val["Age"], bins=[0, 30, 45, 100], labels=["under 30", "30-45", "over 45"])
mask = (val_groups == "under 30").values

for t in [0.3, 0.25, 0.2, 0.15]:
    pred = (probs[mask] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val[mask], pred).ravel()
    print(f"threshold {t}: caught {tp}, missed {fn}, false alarms {fp}")

threshold 0.3: caught 11, missed 3, false alarms 19
threshold 0.25: caught 13, missed 1, false alarms 26
threshold 0.2: caught 14, missed 0, false alarms 30
threshold 0.15: caught 14, missed 0, false alarms 38


In [24]:
thr = np.where(X_test["Age"] <= 30, 0.2, 0.3)
pred_group = (test_probs >= thr).astype(int)

report("Single threshold 0.3 (test)", y_test, (test_probs >= 0.3).astype(int))
report("Per-age threshold (test)", y_test, pred_group)

mask = (X_test["Age"] <= 30).values
tn, fp, fn, tp = confusion_matrix(y_test[mask], pred_group[mask]).ravel()
print(f"Under 30 only: caught {tp}, missed {fn}, false alarms {fp}")

Single threshold 0.3 (test): accuracy 68.8%, recall 92.6%, missed cases 4, false alarms 44
Per-age threshold (test): accuracy 59.7%, recall 94.4%, missed cases 3, false alarms 59
Under 30 only: caught 17, missed 1, false alarms 41


In [25]:
thr = np.where(X_test["Age"] <= 30, 0.2, 0.3)
pred_group = (test_probs >= thr).astype(int)

report("Single threshold 0.3 (test)", y_test, (test_probs >= 0.3).astype(int))
report("Per-group threshold (test)", y_test, pred_group)

def risk_check(age, bmi, blood_pressure, glucose):
    input_data = pd.DataFrame([[age, bmi, blood_pressure, glucose]],
                              columns=["Age", "BMI", "BloodPressure", "Glucose"])
    prob = model.predict_proba(input_data)[:, 1][0]

    threshold = 0.2 if age <= 30 else 0.3

    if prob >= threshold:
        return f"High Risk ({prob:.1%}): Recommend immediate doctor referral & blood test."
    elif prob >= (threshold - 0.1):
        return f"Medium Risk ({prob:.1%}): Suggest lifestyle review and follow-up."
    else:
        return f"Low Risk ({prob:.1%}): Normal indicators."

print(risk_check(age=25, bmi=28.5, blood_pressure=72, glucose=140))

Single threshold 0.3 (test): accuracy 68.8%, recall 92.6%, missed cases 4, false alarms 44
Per-group threshold (test): accuracy 59.7%, recall 94.4%, missed cases 3, false alarms 59
High Risk (42.5%): Recommend immediate doctor referral & blood test.


In [ ]:
import os, re, io, json, time, math, hashlib, secrets, tempfile, datetime, difflib, shutil
import numpy as np
import pandas as pd
import requests
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score, accuracy_score

HEW_ACCESS_CODE = "HEW2026"
DB_PATH = "/content/sugarspot_db.json" if os.path.isdir("/content") else "sugarspot_db.json"
DATA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
FEATURES = ["Age", "BMI", "BloodPressure", "Glucose"]
RANGES = {"Age": (1, 120), "BMI": (10, 70), "BloodPressure": (40, 140), "Glucose": (50, 400)}

COUNTRIES = {
    "Ethiopia": ("251", 9, 9), "Kenya": ("254", 9, 9), "Uganda": ("256", 9, 9),
    "Tanzania": ("255", 9, 9), "Rwanda": ("250", 9, 9), "Nigeria": ("234", 10, 10),
    "Ghana": ("233", 9, 9), "South Africa": ("27", 9, 9), "Egypt": ("20", 10, 10),
    "Morocco": ("212", 9, 9), "Algeria": ("213", 9, 9), "Tunisia": ("216", 8, 8),
    "Sudan": ("249", 9, 9), "South Sudan": ("211", 9, 9), "Somalia": ("252", 8, 9),
    "Djibouti": ("253", 8, 8), "Eritrea": ("291", 7, 7), "Senegal": ("221", 9, 9),
    "Cameroon": ("237", 9, 9), "Cote d'Ivoire": ("225", 10, 10), "Zambia": ("260", 9, 9),
    "Zimbabwe": ("263", 9, 9), "Mozambique": ("258", 9, 9), "Angola": ("244", 9, 9),
    "DR Congo": ("243", 9, 9), "Malawi": ("265", 9, 9), "Botswana": ("267", 7, 8),
    "Namibia": ("264", 9, 9),
    "United States": ("1", 10, 10), "Canada": ("1", 10, 10), "Mexico": ("52", 10, 10),
    "Brazil": ("55", 10, 11), "Argentina": ("54", 10, 10), "Colombia": ("57", 10, 10),
    "Chile": ("56", 9, 9), "Peru": ("51", 9, 9),
    "United Kingdom": ("44", 10, 10), "Ireland": ("353", 9, 9), "Germany": ("49", 10, 11),
    "France": ("33", 9, 9), "Spain": ("34", 9, 9), "Italy": ("39", 9, 10),
    "Portugal": ("351", 9, 9), "Netherlands": ("31", 9, 9), "Belgium": ("32", 9, 9),
    "Switzerland": ("41", 9, 9), "Sweden": ("46", 9, 9), "Norway": ("47", 8, 8),
    "Denmark": ("45", 8, 8), "Finland": ("358", 9, 10), "Poland": ("48", 9, 9),
    "Russia": ("7", 10, 10), "Ukraine": ("380", 9, 9), "Turkey": ("90", 10, 10),
    "Greece": ("30", 10, 10),
    "Saudi Arabia": ("966", 9, 9), "United Arab Emirates": ("971", 9, 9), "Qatar": ("974", 8, 8),
    "Kuwait": ("965", 8, 8), "Oman": ("968", 8, 8), "Bahrain": ("973", 8, 8),
    "Jordan": ("962", 9, 9), "Lebanon": ("961", 7, 8), "Israel": ("972", 9, 9),
    "Iraq": ("964", 10, 10), "Iran": ("98", 10, 10), "Yemen": ("967", 9, 9),
    "India": ("91", 10, 10), "Pakistan": ("92", 10, 10), "Bangladesh": ("880", 10, 10),
    "Sri Lanka": ("94", 9, 9), "Nepal": ("977", 10, 10), "China": ("86", 11, 11),
    "Japan": ("81", 10, 10), "South Korea": ("82", 9, 10), "Vietnam": ("84", 9, 9),
    "Thailand": ("66", 9, 9), "Malaysia": ("60", 9, 10), "Singapore": ("65", 8, 8),
    "Indonesia": ("62", 9, 12), "Philippines": ("63", 10, 10), "Australia": ("61", 9, 9),
    "New Zealand": ("64", 8, 10),
}
COUNTRY_NAMES = sorted(COUNTRIES)
COUNTRY_LABELS = [f"{n} (+{COUNTRIES[n][0]})" for n in COUNTRY_NAMES]
LABEL2NAME = {f"{n} (+{COUNTRIES[n][0]})": n for n in COUNTRY_NAMES}
DEFAULT_LABEL = "Ethiopia (+251)"


def digits_text(mn, mx):
    return f"{mn}" if mn == mx else f"{mn} to {mx}"


def phone_hint(label):
    name = LABEL2NAME.get(label)
    if not name:
        return ""
    dial, mn, mx = COUNTRIES[name]
    return f"📞 **{name}**: dial code **+{dial}** is added automatically. Type **{digits_text(mn, mx)} digits** only."


def clean_phone(label, raw):
    """Returns (full_number, error). Number must match the selected country's digit count."""
    name = LABEL2NAME.get(label)
    if not name:
        return None, "Choose a country for the phone number."
    dial, mn, mx = COUNTRIES[name]
    d = re.sub(r"[\s\-\(\)\.]", "", str(raw or ""))
    if d.startswith("+"):
        d = d[1:]
    if not d:
        return None, "Phone number is required."
    if not d.isdigit():
        return None, "Phone number must contain digits only."
    if d.startswith(dial) and mn <= len(d) - len(dial) <= mx:
        d = d[len(dial):]
    elif d.startswith("0") and mn <= len(d.lstrip("0")) <= mx:
        d = d.lstrip("0")
    if not (mn <= len(d) <= mx):
        return None, (f"{name} numbers need {digits_text(mn, mx)} digits after +{dial} "
                      f"(you typed {len(d)}).")
    return f"+{dial} {d}", None


MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
TODAY_YEAR = datetime.date.today().year
YEARS = list(range(TODAY_YEAR, 1899, -1))
DAYS = list(range(1, 32))
EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(\.[A-Za-z0-9\-]+)*\.[A-Za-z]{2,}$")


def valid_name(s):
    s = (s or "").strip()
    if len(s) < 2 or "  " in s:
        return False
    return all(ch.isalpha() or ch == " " for ch in s) and not s.startswith(" ")


def parse_dob(day, month, year):
    try:
        m = MONTHS.index(month) + 1
        return datetime.date(int(year), m, int(day))
    except Exception:
        return None


def age_from(dob):
    t = datetime.date.today()
    return t.year - dob.year - ((t.month, t.day) < (dob.month, dob.day))


def password_errors(pw):
    e = []
    if not (8 <= len(pw or "") <= 12):
        e.append("8 to 12 characters")
    if not re.search(r"[A-Z]", pw or ""):
        e.append("an uppercase letter")
    if not re.search(r"[a-z]", pw or ""):
        e.append("a lowercase letter")
    if not re.search(r"\d", pw or ""):
        e.append("a number")
    if not re.search(r"[^A-Za-z0-9]", pw or ""):
        e.append("a special character")
    return e


def hash_pw(pw, salt):
    return hashlib.pbkdf2_hmac("sha256", pw.encode(), bytes.fromhex(salt), 100_000).hex()


def now_str():
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M")


def load_db():
    if os.path.exists(DB_PATH):
        try:
            with open(DB_PATH) as f:
                db = json.load(f)
            for k, v in {"users": {}, "threads": {}, "updates": [], "country_data": {}}.items():
                db.setdefault(k, v)
            return db
        except Exception:
            pass
    return {"users": {}, "threads": {}, "updates": [], "country_data": {}}


DB = load_db()


def save_db():
    try:
        with open(DB_PATH, "w") as f:
            json.dump(DB, f)
    except Exception as e:
        print("DB save failed:", e)


RESETS = {}
LOGIN_FAILS = {}


def email_taken(email):
    e = (email or "").strip().lower()
    return any(u["email"].lower() == e for u in DB["users"].values())


def new_id():
    while True:
        uid = "SU-" + "".join(secrets.choice("0123456789") for _ in range(6))
        if uid not in DB["users"]:
            return uid


def load_base_data():
    cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin",
            "BMI", "DiabetesPedigree", "Age", "Outcome"]
    try:
        r = requests.get(DATA_URL, timeout=20)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text), names=cols)
        for c in ["Glucose", "BloodPressure", "BMI"]:
            df[c] = df[c].replace(0, np.nan)
            df[c] = df[c].fillna(df[c].median())
        return df[FEATURES + ["Outcome"]], "Pima Indians Diabetes dataset (real public data)"
    except Exception as e:
        print("Could not download dataset, using synthetic fallback:", e)
        rng = np.random.default_rng(42)
        n = 768
        age = np.clip(rng.lognormal(3.5, 0.3, n), 21, 80)
        bmi = np.clip(rng.normal(32, 7, n), 18, 60)
        bp = np.clip(rng.normal(72, 12, n), 45, 120)
        glu = np.clip(rng.normal(120, 30, n), 60, 250)
        z = -9 + 0.03 * age + 0.07 * bmi + 0.01 * bp + 0.04 * glu
        y = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
        df = pd.DataFrame({"Age": age, "BMI": bmi, "BloodPressure": bp, "Glucose": glu, "Outcome": y})
        return df, "SYNTHETIC FALLBACK data (internet download failed) - for demo only"


def threshold_for(age):
    return 0.2 if age <= 30 else 0.3


def fit_model(df):
    return LogisticRegression(max_iter=1000, class_weight="balanced").fit(df[FEATURES], df["Outcome"])


BASE_DF, DATA_SOURCE = load_base_data()
MEDIANS = {c: float(BASE_DF[c].median()) for c in FEATURES}

from sklearn.linear_model import LinearRegression
_EST_BP = LinearRegression().fit(BASE_DF[["Age", "BMI"]], BASE_DF["BloodPressure"])
_EST_GLU = LinearRegression().fit(BASE_DF[["Age", "BMI"]], BASE_DF["Glucose"])


def estimate_bp_glu(age, bmi):
    x = pd.DataFrame([[age, bmi]], columns=["Age", "BMI"])
    bp = float(np.clip(_EST_BP.predict(x)[0], *RANGES["BloodPressure"]))
    glu = float(np.clip(_EST_GLU.predict(x)[0], *RANGES["Glucose"]))
    return round(bp), round(glu)

_trval, _te = train_test_split(BASE_DF, test_size=0.2, random_state=42, stratify=BASE_DF["Outcome"])
_tr, _val = train_test_split(_trval, test_size=0.25, random_state=42, stratify=_trval["Outcome"])
BASE_MODEL = fit_model(_tr)
_p = BASE_MODEL.predict_proba(_te[FEATURES])[:, 1]
_thr = np.where(_te["Age"] <= 30, 0.2, 0.3)
_pred = (_p >= _thr).astype(int)
_maj = max(_te["Outcome"].mean(), 1 - _te["Outcome"].mean())
METRICS = {
    "n_train": len(_tr), "n_val": len(_val), "n_test": len(_te),
    "recall": recall_score(_te["Outcome"], _pred), "precision": precision_score(_te["Outcome"], _pred),
    "accuracy": accuracy_score(_te["Outcome"], _pred), "baseline_acc": _maj,
}
from sklearn.metrics import confusion_matrix
_y = _te["Outcome"].values
METRICS["tn"], METRICS["fp"], METRICS["fn"], METRICS["tp"] = [int(v) for v in confusion_matrix(_y, _pred).ravel()]
METRICS["n_pos"] = int(_y.sum())
METRICS["groups"] = []
for _name, _mask in [("Age 30 or younger", (_te["Age"] <= 30).values),
                     ("Age 31 to 45", ((_te["Age"] > 30) & (_te["Age"] <= 45)).values),
                     ("Age over 45", (_te["Age"] > 45).values)]:
    _yy, _pp = _y[_mask], _pred[_mask]
    _pos = int(_yy.sum())
    METRICS["groups"].append({
        "name": _name, "n": int(_mask.sum()), "pos": _pos,
        "recall": (float(((_pp == 1) & (_yy == 1)).sum()) / _pos) if _pos else float("nan"),
        "fn": int(((_pp == 0) & (_yy == 1)).sum()), "fp": int(((_pp == 1) & (_yy == 0)).sum()),
    })
_miss = _te[(_pred == 0) & (_y == 1)].assign(Probability=np.round(_p[(_pred == 0) & (_y == 1)], 2)).head(3)
def _results_markdown():
    m = METRICS
    base_acc = m["baseline_acc"]
    out = [f"**Data used for this run:** {DATA_SOURCE}. Trained on {m['n_train']} rows, {m['n_val']} rows used to choose the alert thresholds, "
           f"tested once on {m['n_test']} unseen rows ({m['n_pos']} real cases).", "",
           "| Measure | Baseline (always \"no diabetes\") | Our model |", "|---|---|---|",
           f"| Accuracy | {base_acc:.0%} | {m['accuracy']:.0%} |",
           f"| Recall (real cases caught) | 0% | {m['recall']:.0%} |",
           f"| Precision (alerts that were right) | n/a | {m['precision']:.0%} |",
           f"| **Missed cases (false negatives)** | all {m['n_pos']} real cases | {m['fn']} of {m['n_pos']} |",
           f"| **False alarms (false positives)** | 0 | {m['fp']} |", "",
           "**By age group (test set):** " + "; ".join(
               f"{g['name']}: {g['n']} people, recall {g['recall']:.0%}, missed {g['fn']}, false alarms {g['fp']}"
               for g in m["groups"]) + ".", ""]
    if len(_miss):
        r = _miss.iloc[0]
        out.append(f"**Example the model missed:** a person aged {r['Age']:.0f} with BMI {r['BMI']:.1f}, diastolic blood pressure "
                   f"{r['BloodPressure']:.0f} and glucose {r['Glucose']:.0f} really had diabetes, but the model gave only "
                   f"{r['Probability']:.0%} risk, below the alert threshold, so it said Low risk.")
    else:
        out.append("**Example the model missed:** none on this test set.")
    return "\n".join(out)


print("\n" + "=" * 20 + " COPY EVERYTHING BETWEEN THE LINES INTO THE README (section 'Results') " + "=" * 20)
print(_results_markdown())
print("=" * 108 + "\n")
COUNTRY_MODELS = {}


def retrain_country(country):
    rows = DB["country_data"].get(country, [])
    if len(rows) < 30:
        COUNTRY_MODELS.pop(country, None)
        return None
    cdf = pd.DataFrame(rows)[FEATURES + ["Outcome"]]
    train_df = cdf if len(cdf) >= 200 else pd.concat([BASE_DF, cdf], ignore_index=True)
    COUNTRY_MODELS[country] = (fit_model(train_df), len(cdf))
    return len(cdf)


for _c in list(DB["country_data"]):
    retrain_country(_c)


def get_model(country):
    if country in COUNTRY_MODELS:
        m, n = COUNTRY_MODELS[country]
        return m, f"{country} local model (trained with {n} local records)"
    return BASE_MODEL, ("Global base model (Pima dataset). No local dataset has been uploaded for "
                        f"{country or 'this country'} yet, so results may be less accurate for you")


def predict_prob(model, age, bmi, bp, glu):
    x = pd.DataFrame([[age, bmi, bp, glu]], columns=FEATURES)
    return float(model.predict_proba(x)[0, 1])


def classify(p, age):
    t = threshold_for(age)
    if p >= t:
        return "High", t
    if p >= t - 0.1:
        return "Moderate", t
    return "Low", t


ADVICE = {
    "High": "Your screening result is **above the alert threshold**. Please get a fasting glucose test at a health facility soon and show this result to a health worker.",
    "Moderate": "Your result is **close to the alert threshold**. Consider a glucose check at your next clinic visit and keep healthy habits (activity, less sugary drinks).",
    "Low": "Your result is **below the alert threshold**. Keep healthy habits and re-screen regularly, especially if symptoms appear.",
}


def model_report_md():
    m = METRICS
    return (
        f"**Data:** {DATA_SOURCE}\n\n"
        f"**Model:** Logistic regression (scikit-learn, balanced class weights) on Age, BMI, Blood pressure, Glucose. "
        f"Trained on {m['n_train']} rows, {m['n_val']} rows used to choose thresholds, tested on {m['n_test']} unseen rows.\n\n"
        f"**Alert rule:** flag when probability >= 0.30 (>= 0.20 for age 30 or younger) - tuned to miss fewer real cases.\n\n"
        f"| Measure (test set) | Value |\n|---|---|\n"
        f"| Cases caught (recall) | {m['recall']:.0%} |\n"
        f"| Flags that were right (precision) | {m['precision']:.0%} |\n"
        f"| Accuracy | {m['accuracy']:.0%} |\n"
        f"| **Missed cases (false negatives)** | {m['fn']} of {m['n_pos']} real cases |\n"
        f"| **False alarms (false positives)** | {m['fp']} |\n"
        f"| Baseline 'always say the most common answer' accuracy | {m['baseline_acc']:.0%} (catches 0% of cases) |\n\n"
        + "**By age group (test set):** " + "; ".join(
            f"{g['name']}: recall {g['recall']:.0%}, missed {g['fn']}, false alarms {g['fp']}" for g in m["groups"]) + "\n\n"
        "**Honest limits:** the base data are Pima Indian women aged 21+. The model may be less accurate for men, "
        "children, or other populations. To reduce this bias, a Health Extension Worker can upload a local dataset "
        "for a country (HEW portal > Local dataset) and the model for that country is retrained on it."
    )


MED_INFO = {
    "metformin": "Diabetes medicine that lowers blood sugar; usually taken with meals.",
    "glibenclamide": "Diabetes tablet (sulfonylurea). Can cause low blood sugar if meals are skipped.",
    "glimepiride": "Diabetes tablet (sulfonylurea). Can cause low blood sugar if meals are skipped.",
    "gliclazide": "Diabetes tablet (sulfonylurea). Can cause low blood sugar if meals are skipped.",
    "insulin": "Lowers blood sugar. Dose must follow the prescriber; watch for low blood sugar signs.",
    "sitagliptin": "Diabetes tablet that helps the body release insulin after meals.",
    "pioglitazone": "Diabetes tablet that helps the body respond better to insulin.",
    "empagliflozin": "Diabetes tablet that lowers blood sugar by passing sugar in urine.",
    "dapagliflozin": "Diabetes tablet that lowers blood sugar by passing sugar in urine.",
    "amlodipine": "Blood pressure medicine.",
    "losartan": "Blood pressure medicine.",
    "lisinopril": "Blood pressure medicine.",
    "enalapril": "Blood pressure medicine.",
    "hydrochlorothiazide": "Water pill for blood pressure. In some people it can slightly raise blood sugar.",
    "atorvastatin": "Cholesterol medicine.",
    "simvastatin": "Cholesterol medicine.",
    "aspirin": "Pain / blood-thinning medicine.",
    "prednisolone": "Steroid medicine. Steroids can RAISE blood sugar - tell your health worker.",
    "dexamethasone": "Steroid medicine. Steroids can RAISE blood sugar - tell your health worker.",
}
ALIASES = {"glucophage": "metformin", "daonil": "glibenclamide", "glyburide": "glibenclamide",
           "amaryl": "glimepiride", "diamicron": "gliclazide", "januvia": "sitagliptin",
           "lantus": "insulin", "mixtard": "insulin", "actrapid": "insulin", "humulin": "insulin",
           "norvasc": "amlodipine", "cozaar": "losartan", "lipitor": "atorvastatin",
           "jardiance": "empagliflozin", "forxiga": "dapagliflozin"}
ALL_MED_KEYS = list(MED_INFO) + list(ALIASES)


def med_lookup(name):
    n = (name or "").strip().lower()
    if not n:
        return None, None
    hit = difflib.get_close_matches(n, ALL_MED_KEYS, n=1, cutoff=0.8)
    if not hit:
        return None, None
    k = ALIASES.get(hit[0], hit[0])
    return k, MED_INFO[k]


def ocr_text(img):
    try:
        import pytesseract
    except Exception:
        return None
    if not shutil.which("tesseract"):
        return None
    try:
        return pytesseract.image_to_string(img)
    except Exception:
        return None


def read_medicine_from_photo(img):
    if img is None:
        return gr.update(), "📷 Upload or take a photo of the medicine label first."
    text = ocr_text(img)
    if text is None:
        return gr.update(), "⚠️ Photo reading (OCR) is not installed in this session. Please type the medicine name."
    words = re.findall(r"[A-Za-z]{4,}", text.lower())
    for w in words:
        k, info = med_lookup(w)
        if k:
            return k.title(), f"✅ Found **{k.title()}** on the label (read from your photo). Check it is correct, then press *Save medicine*."
    if words:
        return gr.update(), ("🤔 I could read some text but could not match a known medicine "
                             f"({', '.join(words[:6])}...). Please type the name yourself.")
    return gr.update(), "🤔 I could not read any text. Try a clearer, brighter, closer photo or type the name."


CLIENT_TERMS = """
**SugarSpot - Client Terms & Conditions**

1. **Prototype only.** SugarSpot is a hackathon prototype. It is **not a medical device, not a diagnosis and not medical advice**. Always see a qualified health worker.
2. **AI limits.** The risk score comes from a machine-learning model trained on public data. It can miss real cases (false negatives) and raise false alarms. A normal result does not mean you are healthy.
3. **Who can sign up.** Anyone can create an account. If you are **under 18**, a parent/guardian name, phone and email are required.
4. **What we store.** Name, email, phone, date of birth, your health readings, medicines, habits and messages with HEWs. Photos of medicine labels are read once and **not stored**.
5. **Sharing with HEWs.** Your records are visible to Health Extension Workers **only if you tick the sharing option** (you can change it any time in *Profile*). Messages you send to a HEW are visible to HEWs.
6. **Your ID and password.** You log in with the ID shown after sign-up. Keep your ID and password private. If you forget your password, use *Forgot password* and a verification code.
7. **Location (GPS).** Your location is used only when you press the GPS button, to find nearby facilities. It is not saved.
8. **Voice input.** Voice is handled by your browser's speech service (for example Google in Chrome). Do not speak private details you do not want processed.
9. **Emergencies.** If you feel very unwell (very thirsty, confused, vomiting, fainting), go to the nearest facility immediately - do not wait for this app.
10. **Your data.** You can download your data (CSV/PDF) in *My Data*. You may ask a HEW to delete your account.
11. **Changes.** These terms may change while the prototype is developed.

By ticking the box you confirm you have read and accept these terms.
"""

HEW_TERMS = """
**SugarSpot - Health Extension Worker (HEW) Terms & Conditions**

1. **Professional use only.** Access is for authorised HEWs and their institution, to support community health work.
2. **Confidentiality.** Client information (names, contacts, readings, messages) is confidential. Do not copy, screenshot, share or sell it. Use it only for the client's care.
3. **Consent.** You can see records only for clients who agreed to share. Do not try to access other clients' data.
4. **Medical suggestions.** Suggestions you send are **your own professional judgement**, not AI output. Stay inside your training and refer to a doctor or facility when needed.
5. **Questions to clients.** Ask only questions that are relevant to health screening and care. Be respectful, clear, and use plain language.
6. **Minors.** For clients under 18, involve the parent/guardian in decisions where required by local rules.
7. **Local datasets.** Only upload datasets that are lawful, de-identified and have proper permission. Wrong or biased data will make the model worse.
8. **Access code.** The institution access code is private to your institution. Do not share it.
9. **Prototype.** The system is a prototype; the AI score is a screening aid only and must not replace clinical examination.
10. **Accountability.** Messages you send are logged with your institution name.

By ticking the box you confirm you have read and accept these terms.
"""

def check_email_msg(email):
    e = (email or "").strip()
    if not e:
        return ""
    if not EMAIL_RE.match(e):
        return "❌ That does not look like a valid email."
    if email_taken(e):
        return "❌ This email is already registered. Log in or use another email."
    return "✅ Email is available."


def guardian_visibility(day, month, year):
    dob = parse_dob(day, month, year) if (day and month and year) else None
    return gr.update(visible=bool(dob and age_from(dob) < 18))


def register(first, last, email, country, phone, day, month, year, g_name, g_country, g_phone, g_email,
             pw, pw2, agree, share):
    errs = []
    if not valid_name(first):
        errs.append("First name: letters only (no numbers or symbols).")
    if not valid_name(last):
        errs.append("Last name: letters only (no numbers or symbols).")
    email = (email or "").strip()
    if not EMAIL_RE.match(email):
        errs.append("Enter a valid email.")
    elif email_taken(email):
        errs.append("This email is already registered.")
    full_phone, perr = clean_phone(country, phone)
    if perr:
        errs.append(perr)
    dob = parse_dob(day, month, year)
    guardian = None
    if not dob:
        errs.append("Choose a valid date of birth (day, month and year).")
    elif dob > datetime.date.today():
        errs.append("Date of birth cannot be in the future.")
    elif age_from(dob) < 18:
        if not valid_name(g_name):
            errs.append("Guardian name: letters only.")
        gp, gerr = clean_phone(g_country, g_phone)
        if gerr:
            errs.append("Guardian phone: " + gerr)
        if not EMAIL_RE.match((g_email or "").strip()):
            errs.append("Enter a valid guardian email.")
        if not errs:
            guardian = {"name": g_name.strip(), "phone": gp, "email": g_email.strip()}
    pe = password_errors(pw)
    if pe:
        errs.append("Password needs: " + ", ".join(pe) + ".")
    if pw != pw2:
        errs.append("Passwords do not match.")
    if not agree:
        errs.append("You must accept the Terms & Conditions.")
    if errs:
        return "❌ **Please fix:**\n\n" + "\n".join(f"- {e}" for e in errs), ""
    uid = new_id()
    salt = secrets.token_hex(16)
    DB["users"][uid] = {
        "id": uid, "first": first.strip(), "last": last.strip(), "email": email,
        "country": LABEL2NAME[country], "phone": full_phone, "dob": dob.isoformat(),
        "salt": salt, "hash": hash_pw(pw, salt), "guardian": guardian, "share": bool(share),
        "created": now_str(), "history": [], "medicines": [], "habits": [],
    }
    save_db()
    return (f"✅ **Account created!**\n\n## Your ID: `{uid}`\n\n"
            "⚠️ **Write this ID down.** You need it (and your password) to log in. "
            "Automatic emailing is switched off in this prototype."), uid


def client_login(uid, pw):
    uid = (uid or "").strip().upper()
    key = uid or "?"
    cnt, first = LOGIN_FAILS.get(key, (0, time.time()))
    if cnt >= 5 and time.time() - first < 300:
        return None, "🔒 Too many wrong attempts. Wait 5 minutes and try again."
    u = DB["users"].get(uid)
    if not u or hash_pw(pw or "", u["salt"]) != u["hash"]:
        if time.time() - first > 300:
            cnt, first = 0, time.time()
        LOGIN_FAILS[key] = (cnt + 1, first)
        return None, "❌ Wrong ID or password."
    LOGIN_FAILS.pop(key, None)
    return {"role": "client", "id": uid}, ""


def start_reset(uid, email):
    uid = (uid or "").strip().upper()
    u = DB["users"].get(uid)
    if not u or u["email"].lower() != (email or "").strip().lower():
        return "❌ No account matches that ID and email."
    code = "".join(secrets.choice("0123456789") for _ in range(6))
    RESETS[uid] = {"code": code, "exp": time.time() + 600}
    return (f"✅ Identity matched. Your verification code is **{code}** (valid 10 minutes).\n\n"
            "_Demo mode: the code is shown here because email sending is switched off._")


def finish_reset(uid, code, pw, pw2):
    uid = (uid or "").strip().upper()
    r = RESETS.get(uid)
    if not r or time.time() > r["exp"]:
        return "❌ No active code. Press *Get verification code* first."
    if (code or "").strip() != r["code"]:
        return "❌ Wrong verification code."
    pe = password_errors(pw)
    if pe:
        return "❌ Password needs: " + ", ".join(pe) + "."
    if pw != pw2:
        return "❌ Passwords do not match."
    u = DB["users"][uid]
    u["salt"] = secrets.token_hex(16)
    u["hash"] = hash_pw(pw, u["salt"])
    RESETS.pop(uid, None)
    LOGIN_FAILS.pop(uid, None)
    save_db()
    return "✅ Password changed. Go back and log in with your new password."


def hew_login(inst, code, agree):
    inst = (inst or "").strip()
    if len(inst) < 3 or not re.match(r"^[A-Za-z0-9 .,&'\-]+$", inst):
        return None, "❌ Enter your institution name."
    if not agree:
        return None, "❌ Accept the HEW Terms & Conditions."
    if (code or "") != HEW_ACCESS_CODE:
        return None, "❌ Wrong access code."
    return {"role": "hew", "inst": inst}, ""


HIST_COLS = ["Date", "Age", "BMI", "Blood pressure", "Glucose", "Risk probability", "Risk level", "Notes", "Model"]


def get_user(session):
    if session and session.get("role") == "client":
        return DB["users"].get(session.get("id"))
    return None


def history_df(u):
    rows = [[h["date"], h["age"], h["bmi"], h["bp"], h["glucose"], f"{h['prob']:.0%}", h["level"],
             h.get("notes", ""), h.get("model", "")] for h in (u["history"] if u else [])]
    return pd.DataFrame(rows, columns=HIST_COLS)


def med_df(u):
    rows = [[m["date"], m["name"], m.get("dose", ""), m.get("source", "")] for m in (u["medicines"] if u else [])]
    return pd.DataFrame(rows, columns=["Date", "Medicine", "Dose / note", "Added by"])


def habit_df(u):
    rows = [[h["date"], h["activity"], h["sleep"], h["sugary"], h["smoking"], h["hobbies"]]
            for h in (u["habits"] if u else [])]
    return pd.DataFrame(rows, columns=["Date", "Active days/week", "Sleep hours", "Sugary drinks/day", "Smoking", "Hobbies"])


def render_thread(items):
    if not items:
        return "_No messages yet._"
    out = []
    for m in items:
        who = "🧑 You" if m["from"] == "client" else f"🩺 HEW ({m.get('inst', '')})"
        out.append(f"**{who}** · {m['type']} · _{m['ts']}_\n\n{m['text']}\n\n---")
    return "\n\n".join(out)


def render_updates():
    ups = DB["updates"][::-1]
    if not ups:
        return "_No updates from HEW yet._"
    return "\n\n".join(f"### 📢 {x['title']}\n_{x['inst']} · {x['ts']}_\n\n{x['text']}\n\n---" for x in ups)


def profile_md(u):
    if not u:
        return ""
    dob = datetime.date.fromisoformat(u["dob"])
    s = (f"### 👤 {u['first']} {u['last']}\n\n- **ID:** `{u['id']}`\n- **Email:** {u['email']}\n"
         f"- **Phone:** {u['phone']}\n- **Country:** {u['country']}\n- **Date of birth:** {dob:%d %B %Y} (age {age_from(dob)})\n"
         f"- **Member since:** {u['created']}\n")
    if u.get("guardian"):
        g = u["guardian"]
        s += f"- **Guardian:** {g['name']} · {g['phone']} · {g['email']}\n"
    return s


def client_refresh(session):
    u = get_user(session)
    if not u:
        return ["", "", False, None, history_df(None), "", "", med_df(None), habit_df(None)]
    age = age_from(datetime.date.fromisoformat(u["dob"]))
    return [f"## Welcome, {u['first']} 👋  (ID `{u['id']}`)", profile_md(u), bool(u["share"]), age,
            history_df(u), render_thread(DB["threads"].get(u["id"], [])), render_updates(),
            med_df(u), habit_df(u)]


def save_consent(session, share):
    u = get_user(session)
    if not u:
        return "Please log in again."
    u["share"] = bool(share)
    save_db()
    return "✅ Saved. Your records " + ("**are** shared with HEWs." if share else "are **not** shared with HEWs.")


def wizard_fill(height, weight, symptoms, unk_glu):
    msgs = []
    bmi_val = gr.update()
    try:
        h, w = float(height), float(weight)
        if not (50 <= h <= 250 and 5 <= w <= 300):
            raise ValueError
        bmi = round(w / ((h / 100) ** 2), 1)
        cat = "underweight" if bmi < 18.5 else "healthy range" if bmi < 25 else "overweight" if bmi < 30 else "obesity range"
        bmi_val = bmi
        msgs.append(f"✅ BMI = **{bmi}** ({cat}). Copied into the calculator below.")
    except Exception:
        msgs.append("⚠️ Enter height (50-250 cm) and weight (5-300 kg) to calculate BMI.")
    n = len(symptoms or [])
    if n >= 2:
        msgs.append(f"⚠️ You ticked {n} common warning signs. A glucose test at a facility is strongly recommended"
                    + (" - especially since you don't know your glucose." if unk_glu else "."))
    elif n == 1:
        msgs.append("ℹ️ You ticked 1 warning sign. Keep an eye on it and re-screen.")
    return bmi_val, "\n\n".join(msgs)


def parse_voice(text):
    nums = re.findall(r"\d+(?:[.,]\d+)?", str(text or ""))
    vals = []
    for n in nums[:4]:
        try:
            vals.append(float(n.replace(",", ".")))
        except Exception:
            vals.append(None)
    while len(vals) < 4:
        vals.append(None)
    if not nums:
        return [gr.update()] * 4 + ["🤔 I heard no numbers. Say 4 numbers: **age, BMI, blood pressure, glucose**."]
    note = f"✅ Heard: {', '.join(nums[:4])}. Filled in order: age, BMI, blood pressure, glucose. Please check them."
    return [v if v is not None else gr.update() for v in vals] + [note]


def result_card(p, level, thr, model_label, estimated, age):
    color = {"High": "#c62828", "Moderate": "#ef6c00", "Low": "#2e7d32"}[level]
    est = ""
    if estimated:
        est = ("<p>⚠️ <b>Estimated (not measured):</b> " +
               " · ".join(f"{n}: <b>{v}</b>" for n, v in estimated) +
               ". They are predicted from your age and BMI, so confidence is lower - please measure them.</p>")
    young = ("<p>ℹ️ The base model was trained on adults 21+, so it may be less accurate at your age.</p>"
             if age < 21 else "")
    adv = ADVICE[level].replace("**", "")
    return (f"<div style='background:{color};color:#fff;border-radius:12px;padding:16px'>"
            f"<h2 style='margin:0'>Risk level: {level}</h2>"
            f"<h3 style='margin:4px 0'>Risk probability: {p:.0%} (alert at {thr:.0%})</h3>"
            f"<p>{adv}</p>{est}{young}"
            f"<p style='font-size:0.85em;opacity:.9'>Model used: {model_label}</p></div>")


SPEECH = {
    "en": ("Diabetes risk level", "high", "moderate", "low", "Risk probability", "percent", "This is a prototype, not medical advice."),
    "am": ("የስኳር በሽታ ተጋላጭነት ደረጃ", "ከፍተኛ", "መካከለኛ", "ዝቅተኛ", "የተጋላጭነት ዕድል", "በመቶ", "ይህ የሙከራ ፕሮቶታይፕ ነው፣ የሕክምና ምክር አይደለም።"),
    "ar": ("مستوى خطر السكري", "مرتفع", "متوسط", "منخفض", "احتمال الخطر", "بالمئة", "هذا نموذج أولي وليس نصيحة طبية."),
    "fr": ("Niveau de risque de diabète", "élevé", "modéré", "faible", "Probabilité de risque", "pour cent", "Ceci est un prototype, pas un avis médical."),
    "es": ("Nivel de riesgo de diabetes", "alto", "moderado", "bajo", "Probabilidad de riesgo", "por ciento", "Esto es un prototipo, no un consejo médico."),
    "pt": ("Nível de risco de diabetes", "alto", "moderado", "baixo", "Probabilidade de risco", "por cento", "Isto é um protótipo, não é aconselhamento médico."),
    "de": ("Diabetes-Risikostufe", "hoch", "mittel", "niedrig", "Risikowahrscheinlichkeit", "Prozent", "Dies ist ein Prototyp und keine medizinische Beratung."),
    "sw": ("Kiwango cha hatari ya kisukari", "juu", "wastani", "chini", "Uwezekano wa hatari", "asilimia", "Huu ni mfano wa majaribio, si ushauri wa kitabibu."),
    "hi": ("मधुमेह जोखिम स्तर", "उच्च", "मध्यम", "कम", "जोखिम की संभावना", "प्रतिशत", "यह केवल एक प्रोटोटाइप है, चिकित्सा सलाह नहीं।"),
    "zh-CN": ("糖尿病风险等级", "高", "中等", "低", "风险概率", "百分之", "这只是原型，不是医疗建议。"),
    "ja": ("糖尿病リスクレベル", "高い", "中程度", "低い", "リスク確率", "パーセント", "これは試作品であり、医療アドバイスではありません。"),
    "ko": ("당뇨병 위험 수준", "높음", "보통", "낮음", "위험 확률", "퍼센트", "이것은 시제품이며 의료 조언이 아닙니다."),
    "ru": ("Уровень риска диабета", "высокий", "умеренный", "низкий", "Вероятность риска", "процентов", "Это прототип, а не медицинский совет."),
    "tr": ("Diyabet risk düzeyi", "yüksek", "orta", "düşük", "Risk olasılığı", "yüzde", "Bu bir prototiptir, tıbbi tavsiye değildir."),
    "it": ("Livello di rischio di diabete", "alto", "moderato", "basso", "Probabilità di rischio", "per cento", "Questo è un prototipo, non un consiglio medico."),
}


def speak_result(level, p, lang_code):
    """Spoken risk result (mp3) in the chosen language. Returns (audio_path or None, note)."""
    code = lang_code if lang_code in SPEECH else "en"
    T = SPEECH[code]
    word = {"High": T[1], "Moderate": T[2], "Low": T[3]}[level]
    text = f"{T[0]}: {word}. {T[4]}: {round(p * 100)} {T[5]}. {T[6]}"
    note = "" if code == lang_code else "ℹ️ Voice for this language is not available yet - spoken in English."
    try:
        try:
            from gtts import gTTS
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gTTS"], check=False)
            from gtts import gTTS
        path = os.path.join(tempfile.gettempdir(), f"sugarspot_voice_{int(time.time() * 1000)}.mp3")
        gTTS(text=text, lang=code).save(path)
        return path, note
    except Exception as e:
        return None, f"⚠️ Voice output needs internet (Google text-to-speech) and could not run: {e}"


def assess(age, bmi, bp, glu, unk_bp, unk_glu, session, lang="en"):
    u = get_user(session)
    if not u:
        return "<b>Please log in again.</b>", gr.update(), gr.update(), None, ""

    def warn(k, lo, hi):
        return (f"<div style='background:#ef6c00;color:#fff;padding:12px;border-radius:10px'>"
                f"⚠️ {k} must be between {lo} and {hi}.</div>"), gr.update(), gr.update(), None, ""

    for k, v in (("Age", age), ("BMI", bmi)):
        lo, hi = RANGES[k]
        if v is None or not (lo <= float(v) <= hi):
            return warn(k, lo, hi)
    estimated, bp_out, glu_out = [], gr.update(), gr.update()
    if unk_bp or unk_glu:
        e_bp, e_glu = estimate_bp_glu(float(age), float(bmi))
        if unk_bp:
            bp, bp_out = e_bp, e_bp
            estimated.append(("Blood pressure", f"{e_bp} mmHg"))
        if unk_glu:
            glu, glu_out = e_glu, e_glu
            estimated.append(("Glucose", f"{e_glu} mg/dL"))
    for k, v in (("BloodPressure", bp), ("Glucose", glu)):
        lo, hi = RANGES[k]
        if v is None or not (lo <= float(v) <= hi):
            return warn(k, lo, hi)
    model, label = get_model(u["country"])
    p = predict_prob(model, float(age), float(bmi), float(bp), float(glu))
    level, thr = classify(p, float(age))
    u["history"].append({"date": now_str(), "age": float(age), "bmi": float(bmi), "bp": float(bp),
                         "glucose": float(glu), "prob": p, "level": level,
                         "notes": ("estimated: " + ", ".join(f"{n} {v}" for n, v in estimated)) if estimated else "",
                         "model": label[:40]})
    save_db()
    audio, note = speak_result(level, p, lang)
    return result_card(p, level, thr, label, estimated, float(age)), bp_out, glu_out, audio, note


def add_medicine(name, dose, from_photo, session):
    u = get_user(session)
    if not u:
        return "Please log in again.", med_df(None)
    name = (name or "").strip()
    if not name:
        return "⚠️ Type the medicine name or read it from a photo first.", med_df(u)
    k, info = med_lookup(name)
    u["medicines"].append({"date": now_str(), "name": name, "dose": (dose or "").strip(),
                           "source": "photo" if from_photo else "typed"})
    save_db()
    note = f"✅ Saved **{name}**."
    if info:
        note += f"\n\nℹ️ {info}  \n_General information only - follow your prescriber._"
    else:
        note += "\n\nℹ️ No information for this name in the small built-in list."
    return note, med_df(u)


def add_habits(act, sleep, sugary, smoking, hobbies, session):
    u = get_user(session)
    if not u:
        return "Please log in again.", habit_df(None)
    u["habits"].append({"date": now_str(), "activity": act, "sleep": sleep, "sugary": sugary,
                        "smoking": smoking, "hobbies": (hobbies or "").strip()})
    save_db()
    tips = []
    if act < 3:
        tips.append("Try to be active on most days of the week (around 150 minutes weekly).")
    if sugary >= 2:
        tips.append("Cutting down sugary drinks helps blood sugar.")
    if sleep < 6:
        tips.append("Short sleep can affect blood sugar; aim for 7-9 hours.")
    if smoking == "Current":
        tips.append("Stopping smoking lowers diabetes complications.")
    return "✅ Saved. " + (" ".join(tips) if tips else "Great habits - keep it up!"), habit_df(u)


def make_csv(session):
    u = get_user(session)
    if not u:
        return None
    path = os.path.join(tempfile.gettempdir(), f"sugarspot_{u['id']}.csv")
    history_df(u).to_csv(path, index=False)
    return path


def make_pdf(session):
    u = get_user(session)
    if not u:
        return None
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas
    path = os.path.join(tempfile.gettempdir(), f"sugarspot_{u['id']}.pdf")
    c = canvas.Canvas(path, pagesize=letter)
    W, H = letter
    y = H - 50

    def line(t, size=10, bold=False):
        nonlocal y
        if y < 60:
            c.showPage()
            y = H - 50
        c.setFont("Helvetica-Bold" if bold else "Helvetica", size)
        c.drawString(40, y, t[:110])
        y -= size + 5

    line("SugarSpot - My Health Data", 16, True)
    line("CAUTION: used only as a prototype and not as a medical advice", 10, True)
    y -= 6
    dob = datetime.date.fromisoformat(u["dob"])
    for t in [f"Name: {u['first']} {u['last']}", f"ID: {u['id']}", f"Email: {u['email']}",
              f"Phone: {u['phone']}", f"Country: {u['country']}", f"Date of birth: {dob:%d %B %Y}"]:
        line(t)
    y -= 6
    line("Screening history", 12, True)
    for h in u["history"] or []:
        line(f"{h['date']} | age {h['age']:.0f} BMI {h['bmi']} BP {h['bp']} Glucose {h['glucose']} | "
             f"{h['prob']:.0%} | {h['level']}")
    if not u["history"]:
        line("No screenings yet.")
    y -= 6
    line("Medicines", 12, True)
    for m in u["medicines"] or []:
        line(f"{m['date']} | {m['name']} {m.get('dose', '')}")
    if not u["medicines"]:
        line("None saved.")
    y -= 6
    line("Habits", 12, True)
    for h in u["habits"] or []:
        line(f"{h['date']} | active {h['activity']}d/wk, sleep {h['sleep']}h, sugary {h['sugary']}/day, "
             f"smoking {h['smoking']}, hobbies {h['hobbies']}")
    if not u["habits"]:
        line("None saved.")
    c.save()
    return path


def send_client_msg(mtype, text, session):
    u = get_user(session)
    if not u:
        return "Please log in again.", ""
    text = (text or "").strip()
    if not text:
        return "⚠️ Write a message first.", render_thread(DB["threads"].get(u["id"], []))
    DB["threads"].setdefault(u["id"], []).append(
        {"from": "client", "type": mtype, "text": text, "ts": now_str()})
    save_db()
    return "", render_thread(DB["threads"][u["id"]])


def refresh_messages(session):
    u = get_user(session)
    if not u:
        return "", ""
    return render_thread(DB["threads"].get(u["id"], [])), render_updates()


def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


FALLBACK_FACILITIES = [
    ("Tikur Anbessa (Black Lion) Hospital", "hospital", 9.0343, 38.7490),
    ("St. Paul's Hospital Millennium Medical College", "hospital", 9.0113, 38.7186),
    ("Zewditu Memorial Hospital", "hospital", 9.0105, 38.7612),
]


def fetch_facilities(lat, lon, radius_km):
    q = (f'[out:json][timeout:25];(nwr["amenity"~"hospital|clinic|doctors"](around:{int(radius_km * 1000)},{lat},{lon});'
         f'nwr["healthcare"~"hospital|clinic|centre"](around:{int(radius_km * 1000)},{lat},{lon}););out center 80;')
    r = requests.post("https://overpass-api.de/api/interpreter", data={"data": q}, timeout=30,
                      headers={"User-Agent": "sugarSpot-hackathon-prototype"})
    r.raise_for_status()
    out, seen = [], set()
    for el in r.json().get("elements", []):
        tags = el.get("tags", {})
        la = el.get("lat") or (el.get("center") or {}).get("lat")
        lo = el.get("lon") or (el.get("center") or {}).get("lon")
        if la is None or lo is None:
            continue
        name = tags.get("name") or tags.get("name:en") or "Unnamed health facility"
        kind = tags.get("amenity") or tags.get("healthcare") or "facility"
        key = (name, round(la, 4), round(lo, 4))
        if key in seen:
            continue
        seen.add(key)
        out.append((name, kind, la, lo))
    return out, True


def find_facilities(lat, lon, radius, ftype, session):
    try:
        lat, lon = float(lat), float(lon)
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            raise ValueError
    except Exception:
        return "", "⚠️ Press **Use my GPS location** (and allow location) or type valid latitude and longitude."
    bbox = f"{lon - 0.05},{lat - 0.04},{lon + 0.05},{lat + 0.04}"
    map_html = (f"<iframe width='100%' height='320' style='border:0;border-radius:10px' "
                f"src='https://www.openstreetmap.org/export/embed.html?bbox={bbox}&layer=mapnik&marker={lat},{lon}'></iframe>")
    note, live = "", True
    try:
        facs, _ = fetch_facilities(lat, lon, radius)
        if not facs and radius < 25:
            facs, _ = fetch_facilities(lat, lon, 25)
            note = "No facility within your radius, searched 25 km instead. "
    except Exception:
        facs, live = list(FALLBACK_FACILITIES), False
        note = "⚠️ Live map service unreachable - showing an approximate sample list. "
    u = get_user(session)
    level = u["history"][-1]["level"] if u and u["history"] else "Unknown"
    rows = []
    for name, kind, la, lo in facs:
        k = "hospital" if "hospital" in kind else "clinic"
        if ftype == "Hospitals" and k != "hospital":
            continue
        if ftype == "Clinics / health centres" and k == "hospital":
            continue
        d = haversine(lat, lon, la, lo)
        score = d - (3.0 if (level == "High" and k == "hospital") else 0)
        rows.append((score, name, k, d, la, lo))
    rows.sort()
    if not rows:
        return map_html, "😕 No facilities found for this filter. Try a bigger radius or 'All'."
    why = ("Your last result was <b>High</b>, so hospitals are ranked a bit higher." if level == "High"
           else "Ranked by distance.")
    html = [f"<p>{note}{why}</p>",
            "<table style='width:100%;border-collapse:collapse'><tr style='text-align:left'>"
            "<th>#</th><th>Facility</th><th>Type</th><th>Distance</th><th>Directions</th></tr>"]
    for i, (_, name, k, d, la, lo) in enumerate(rows[:10], 1):
        link = f"https://www.google.com/maps/dir/?api=1&destination={la},{lo}"
        html.append(f"<tr><td>{i}</td><td>{name}</td><td>{k}</td><td>{d:.1f} km</td>"
                    f"<td><a href='{link}' target='_blank'>Open route</a></td></tr>")
    html.append("</table>")
    return map_html, "".join(html)


def hew_records(country, sort):
    rows = []
    order = {"High": 0, "Moderate": 1, "Low": 2, "-": 3}
    for u in DB["users"].values():
        if not u["share"]:
            continue
        if country and country != "All countries" and u["country"] != country:
            continue
        last = u["history"][-1] if u["history"] else None
        age = age_from(datetime.date.fromisoformat(u["dob"]))
        rows.append([u["id"], f"{u['first']} {u['last']}", u["country"], age, u["phone"],
                     last["level"] if last else "-", f"{last['prob']:.0%}" if last else "-",
                     last["date"] if last else "-", len(u["history"])])
    cols = ["ID", "Name", "Country", "Age", "Phone", "Latest risk", "Probability", "Last screening", "# screenings"]
    df = pd.DataFrame(rows, columns=cols)
    if len(df):
        if sort == "Highest risk first":
            df = df.sort_values(by="Latest risk", key=lambda s: s.map(order))
        elif sort == "Newest screening first":
            df = df.sort_values(by="Last screening", ascending=False)
        elif sort == "Country A-Z":
            df = df.sort_values(by="Country")
    countries = ["All countries"] + sorted({u["country"] for u in DB["users"].values()})
    return df, gr.update(choices=countries)


def hew_client_choices():
    ids = []
    for u in DB["users"].values():
        if u["share"] or u["id"] in DB["threads"]:
            ids.append((f"{u['id']} - {u['first']} {u['last']}", u["id"]))
    return gr.update(choices=ids)


def hew_load_thread(cid):
    if not cid or cid not in DB["users"]:
        return "_Select a client._"
    u = DB["users"][cid]
    head = f"**{u['first']} {u['last']}** · {u['country']}"
    if u["history"]:
        h = u["history"][-1]
        head += f" · latest: {h['level']} ({h['prob']:.0%}) on {h['date']}"
    return head + "\n\n---\n\n" + render_thread(DB["threads"].get(cid, []))


def hew_send(session, cid, mtype, text):
    if not session or session.get("role") != "hew":
        return "Please log in again.", ""
    if not cid or cid not in DB["users"]:
        return "⚠️ Select a client first.", ""
    text = (text or "").strip()
    if not text:
        return "⚠️ Write a message first.", hew_load_thread(cid)
    DB["threads"].setdefault(cid, []).append(
        {"from": "hew", "inst": session["inst"], "type": mtype, "text": text, "ts": now_str()})
    save_db()
    return "✅ Sent.", hew_load_thread(cid)


def hew_post_update(session, title, text):
    if not session or session.get("role") != "hew":
        return "Please log in again.", ""
    if not (title or "").strip() or not (text or "").strip():
        return "⚠️ Add a title and a message.", render_updates()
    DB["updates"].append({"title": title.strip(), "text": text.strip(), "inst": session["inst"], "ts": now_str()})
    save_db()
    return "✅ Update published to all clients.", render_updates()


def hew_upload_dataset(session, country, path):
    if not session or session.get("role") != "hew":
        return "Please log in again."
    if not country or not path:
        return "⚠️ Choose a country and a CSV file."
    try:
        df = pd.read_csv(path)
    except Exception as e:
        return f"❌ Could not read the CSV: {e}"
    need = FEATURES + ["Outcome"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        return f"❌ Missing columns: {', '.join(missing)}. Required: {', '.join(need)} (Outcome = 0 or 1)."
    df = df[need].apply(pd.to_numeric, errors="coerce").dropna()
    for k, (lo, hi) in RANGES.items():
        df = df[(df[k] >= lo) & (df[k] <= hi)]
    df = df[df["Outcome"].isin([0, 1])]
    if len(df) < 30:
        return f"❌ Only {len(df)} valid rows after cleaning. Need at least 30."
    if df["Outcome"].nunique() < 2:
        return "❌ The data need both Outcome 0 and Outcome 1 rows."
    DB["country_data"].setdefault(country, [])
    DB["country_data"][country] += df.to_dict("records")
    save_db()
    n = retrain_country(country)
    return (f"✅ Added {len(df)} valid rows for **{country}** (total {n}). The model for {country} was retrained. "
            "New screenings for clients in that country now use it.")


def local_models_md():
    if not COUNTRY_MODELS:
        return "_No local models yet - only the global base model is active._"
    return "\n".join(f"- **{c}**: {n} local records" for c, (_, n) in COUNTRY_MODELS.items())


VOICE_LANGS = [("English", "en-US"), ("Amharic (አማርኛ)", "am-ET"), ("Swahili", "sw-KE"), ("Arabic", "ar-SA"),
               ("French", "fr-FR"), ("Spanish", "es-ES"), ("Portuguese", "pt-BR"), ("German", "de-DE"),
               ("Hindi", "hi-IN"), ("Chinese (Mandarin)", "zh-CN"), ("Japanese", "ja-JP"),
               ("Korean", "ko-KR"), ("Russian", "ru-RU"), ("Turkish", "tr-TR"), ("Italian", "it-IT")]

VOICE_JS = """
(lang) => {
  const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
  if (!SR) { alert('Voice input needs Google Chrome or Microsoft Edge.'); return; }
  const r = new SR(); r.lang = lang; r.interimResults = false; r.maxAlternatives = 1;
  r.onresult = (e) => {
    const t = e.results[0][0].transcript;
    const el = document.querySelector('#voice_box textarea') || document.querySelector('#voice_box input');
    if (el) { el.value = t; el.dispatchEvent(new Event('input', {bubbles: true})); }
  };
  r.onerror = (e) => alert('Voice error: ' + e.error + '. Allow the microphone and try again.');
  r.start();
}
"""

GPS_JS = """
async () => {
  if (!navigator.geolocation) { alert('GPS is not supported on this browser.'); return ['', '']; }
  return await new Promise((res) => navigator.geolocation.getCurrentPosition(
    (p) => res([String(p.coords.latitude), String(p.coords.longitude)]),
    (e) => { alert('Location blocked: ' + e.message + '. Allow location for this page, or type the coordinates.'); res(['', '']); },
    {enableHighAccuracy: true, timeout: 15000}));
}
"""

CSS = """
.caution{background:#c62828;color:#fff;padding:10px 14px;border-radius:8px;font-weight:700;text-align:center;margin-bottom:8px}
.bigbtn button{font-size:1.1em;padding:14px}
"""
CAUTION_HTML = '<div class="caution">🚨 CAUTION: used only as a prototype and not as a medical advice</div>'

LANG_CHOICES = [("English", "en"), ("አማርኛ (Amharic)", "am"), ("Afaan Oromoo", "om"), ("ትግርኛ (Tigrinya)", "ti"),
                ("العربية (Arabic)", "ar"), ("Français", "fr"), ("Español", "es"), ("Português", "pt"),
                ("Deutsch", "de"), ("Kiswahili", "sw"), ("हिन्दी (Hindi)", "hi"), ("中文 (Chinese)", "zh-CN"),
                ("日本語", "ja"), ("한국어", "ko"), ("Русский", "ru"), ("Türkçe", "tr"), ("Italiano", "it")]
LANG_TO_VOICE = {"en": "en-US", "am": "am-ET", "ar": "ar-SA", "fr": "fr-FR", "es": "es-ES", "pt": "pt-BR",
                 "de": "de-DE", "sw": "sw-KE", "hi": "hi-IN", "zh-CN": "zh-CN", "ja": "ja-JP", "ko": "ko-KR",
                 "ru": "ru-RU", "tr": "tr-TR", "it": "it-IT"}
_INCLUDED = ",".join(code for _, code in LANG_CHOICES)

HEAD_HTML = """
<script src="https://translate.google.com/translate_a/element.js"></script>
<style>
  .goog-te-banner-frame, .skiptranslate iframe, #goog-gt-tt, .goog-tooltip{display:none !important}
  body{top:0 !important}
  .goog-text-highlight{background:none !important; box-shadow:none !important}
</style>
"""

LANG_JS = """
(code) => {
  const INCLUDED = '%s';
  const boot = () => {
    if (!document.getElementById('google_translate_element')) {
      const d = document.createElement('div');
      d.id = 'google_translate_element'; d.style.display = 'none';
      document.body.appendChild(d);
    }
    if (window.google && google.translate && google.translate.TranslateElement
        && !document.querySelector('select.goog-te-combo')) {
      new google.translate.TranslateElement(
        {pageLanguage: 'en', autoDisplay: false, includedLanguages: INCLUDED}, 'google_translate_element');
    }
  };
  const apply = () => {
    const sel = document.querySelector('select.goog-te-combo');
    if (!sel) return false;
    sel.value = code;
    sel.dispatchEvent(new Event('change'));
    return true;
  };
  boot();
  let tries = 0;
  const timer = setInterval(() => { boot(); if (apply() || ++tries > 50) clearInterval(timer); }, 300);
}
""" % _INCLUDED

LOGO_SVG = """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 560 160" role="img" aria-label="SugarSpot logo" style="max-width:420px;width:100%;height:auto">
  <defs>
    <linearGradient id="ssdrop" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#ff6f61"/><stop offset="1" stop-color="#d32f2f"/></linearGradient>
    <linearGradient id="sswing" x1="0" y1="0" x2="1" y2="0"><stop offset="0" stop-color="#26c6da"/><stop offset="1" stop-color="#00838f"/></linearGradient>
    <g id="ssw">
      <path d="M126 74 Q150 40 180 42 Q160 62 126 80 Z" fill="url(#sswing)"/>
      <path d="M126 83 Q152 62 178 68 Q158 82 126 90 Z" fill="url(#sswing)" opacity=".85"/>
      <path d="M126 92 Q148 82 170 92 Q152 102 126 98 Z" fill="url(#sswing)" opacity=".7"/>
    </g>
  </defs>
  <use href="#ssw"/><use href="#ssw" transform="translate(190,0) scale(-1,1)"/>
  <path d="M95 22 C95 22 62 60 62 88 A33 33 0 0 0 128 88 C128 60 95 22 95 22 Z" fill="url(#ssdrop)"/>
  <circle cx="84" cy="96" r="8" fill="#fff" opacity=".9"/>
  <path d="M107 70 q8 8 8 20" stroke="#fff" stroke-width="4" stroke-linecap="round" fill="none" opacity=".55"/>
  <text translate="no" x="192" y="92" font-family="Arial, Helvetica, sans-serif" font-size="60" font-weight="700" fill="#1a237e">Sugar<tspan fill="#d32f2f">Spot</tspan></text>
</svg>"""
SLOGAN = "Wings to lift, insight to guide, SugarSpot to heal."
HEADER_HTML = (f"<div style='display:flex;flex-direction:column;align-items:center;margin:4px 0 8px'>{LOGO_SVG}"
               f"<div style='font-style:italic;color:#546e7a;font-size:1.05em;text-align:center'>{SLOGAN}</div></div>")

try:
    open("SugarSpot_logo.svg", "w").write(LOGO_SVG.replace(' style="max-width:420px;width:100%;height:auto"', "")
        .replace("</svg>", f'<text x="193" y="122" font-family="Arial, Helvetica, sans-serif" font-size="14" font-style="italic" fill="#546e7a">{SLOGAN}</text></svg>'))
except Exception:
    pass

VIEW_NAMES = ["landing", "signup", "login", "forgot", "hewlogin", "app", "hewapp"]


def go(name):
    return [gr.update(visible=(n == name)) for n in VIEW_NAMES]


def toggle_pw(show):
    return gr.update(type="text" if show else "password")


try:
    GR_MAJOR = int(gr.__version__.split(".")[0])
except Exception:
    GR_MAJOR = 5
_BLOCKS_KW = {} if GR_MAJOR >= 6 else dict(css=CSS, theme=gr.themes.Soft(), head=HEAD_HTML)
_LAUNCH_KW = dict(css=CSS, theme=gr.themes.Soft(), head=HEAD_HTML) if GR_MAJOR >= 6 else {}
print("Gradio version:", getattr(gr, "__version__", "?"))

with gr.Blocks(title="SugarSpot", **_BLOCKS_KW) as demo:
    session = gr.State({})
    new_account_id = gr.State("")
    gr.HTML(CAUTION_HTML)
    gr.HTML(HEADER_HTML)
    lang_dd = gr.Dropdown(LANG_CHOICES, value="en", label="🌐 Language / ቋንቋ (the whole page is translated automatically)")

    with gr.Column(visible=True) as v_landing:
        gr.Markdown("### Who are you?")
        b_client = gr.Button("👤 I am a Client", variant="primary", elem_classes="bigbtn")
        b_hew = gr.Button("🩺 I am a Health Extension Worker (HEW)", elem_classes="bigbtn")

    with gr.Column(visible=False) as v_signup:
        gr.Markdown("### Create your client account")
        with gr.Row():
            su_first = gr.Textbox(label="First name (letters only)")
            su_last = gr.Textbox(label="Last name (letters only)")
        su_email = gr.Textbox(label="Email")
        su_email_status = gr.Markdown()
        su_country = gr.Dropdown(COUNTRY_LABELS, value=DEFAULT_LABEL, label="Country / dial code")
        su_phone_hint = gr.Markdown(phone_hint(DEFAULT_LABEL))
        su_phone = gr.Textbox(label="Phone number (without the dial code)")
        gr.Markdown("**Date of birth**")
        with gr.Row():
            su_day = gr.Dropdown(DAYS, label="Day")
            su_month = gr.Dropdown(MONTHS, label="Month")
            su_year = gr.Dropdown(YEARS, label="Year")
        with gr.Group(visible=False) as su_guardian:
            gr.Markdown("**You are under 18 - parent/guardian details are required**")
            g_name = gr.Textbox(label="Guardian full name (letters only)")
            g_country = gr.Dropdown(COUNTRY_LABELS, value=DEFAULT_LABEL, label="Guardian country / dial code")
            g_phone_hint = gr.Markdown(phone_hint(DEFAULT_LABEL))
            g_phone = gr.Textbox(label="Guardian phone (without the dial code)")
            g_email = gr.Textbox(label="Guardian email")
        su_pw = gr.Textbox(label="Password (8-12 chars: upper, lower, number, special)", type="password")
        su_pw_show = gr.Checkbox(label="👁 Show password")
        su_pw2 = gr.Textbox(label="Confirm password", type="password")
        with gr.Accordion("📄 Read the Terms & Conditions", open=False):
            gr.Markdown(CLIENT_TERMS)
        su_agree = gr.Checkbox(label="I have read and accept the Terms & Conditions")
        su_share = gr.Checkbox(label="(Optional) Share my records with Health Extension Workers", value=False)
        su_msg = gr.Markdown()
        su_submit = gr.Button("Create account", variant="primary")
        with gr.Group(visible=False) as su_done:
            gr.Markdown("### 🔑 Your ID number - copy it now")
            su_id_box = gr.Textbox(label="Your ID number", interactive=False)
            su_copy = gr.Button("📋 Copy my ID")
            su_confirm = gr.Checkbox(label="✅ I have copied my ID and saved it somewhere safe")
            su_open_msg = gr.Markdown()
            su_open = gr.Button("Open my account ➜", variant="primary")
        su_back = gr.Button("⬅ Back")

    with gr.Column(visible=False) as v_login:
        gr.Markdown("### Client login")
        lg_id = gr.Textbox(label="Your ID number (example: SU-123456)")
        lg_pw = gr.Textbox(label="Password", type="password")
        lg_show = gr.Checkbox(label="👁 Show password")
        lg_msg = gr.Markdown()
        lg_btn = gr.Button("Log in", variant="primary")
        with gr.Row():
            lg_forgot = gr.Button("Forgot your password?")
            lg_signup = gr.Button("Create new account")
        lg_back = gr.Button("⬅ Back")

    with gr.Column(visible=False) as v_forgot:
        gr.Markdown("### Restore your password")
        gr.Markdown("Step 1: enter your ID and the email you signed up with.")
        fg_id = gr.Textbox(label="Your ID number")
        fg_email = gr.Textbox(label="Email used at sign-up")
        fg_get = gr.Button("Get verification code")
        fg_msg1 = gr.Markdown()
        gr.Markdown("Step 2: enter the code and choose a new password.")
        fg_code = gr.Textbox(label="Verification code")
        fg_pw = gr.Textbox(label="New password", type="password")
        fg_show = gr.Checkbox(label="👁 Show password")
        fg_pw2 = gr.Textbox(label="Confirm new password", type="password")
        fg_set = gr.Button("Change password", variant="primary")
        fg_msg2 = gr.Markdown()
        fg_back = gr.Button("⬅ Back to login")

    with gr.Column(visible=False) as v_hewlogin:
        gr.Markdown("### Health Extension Worker access")
        hw_inst = gr.Textbox(label="Institution name")
        hw_code = gr.Textbox(label="Access code", type="password")
        hw_show = gr.Checkbox(label="👁 Show code")
        with gr.Accordion("📄 Read the HEW Terms & Conditions", open=False):
            gr.Markdown(HEW_TERMS)
        hw_agree = gr.Checkbox(label="I have read and accept the HEW Terms & Conditions")
        hw_msg = gr.Markdown()
        hw_btn = gr.Button("Enter HEW portal", variant="primary")
        hw_back = gr.Button("⬅ Back")

    with gr.Column(visible=False) as v_app:
        with gr.Row():
            welcome = gr.Markdown()
            signout = gr.Button("🚪 Sign out", scale=0)
        with gr.Tabs():
            with gr.Tab("🏠 Home") as t_home:
                gr.Markdown("### Step 1 - Smart wizard (don't know your numbers? start here)")
                with gr.Row():
                    wz_h = gr.Number(label="Height (cm)")
                    wz_w = gr.Number(label="Weight (kg)")
                wz_sym = gr.CheckboxGroup(
                    ["Very thirsty", "Urinating a lot", "Always tired", "Blurred vision",
                     "Slow-healing wounds", "Family member with diabetes"],
                    label="Do you notice any of these?")
                with gr.Row():
                    unk_bp = gr.Checkbox(label="I don't know my blood pressure")
                    unk_glu = gr.Checkbox(label="I don't know my glucose")
                wz_btn = gr.Button("Calculate BMI & check warning signs")
                wz_msg = gr.Markdown()
                gr.Markdown("### Step 2 - Risk calculator")
                with gr.Accordion("🎤 Hands-free voice input (any language your browser supports)", open=False):
                    voice_lang = gr.Dropdown(VOICE_LANGS, value="en-US", label="Speaking language")
                    mic_btn = gr.Button("🎤 Speak: age, BMI, blood pressure, glucose")
                    voice_box = gr.Textbox(label="What I heard", elem_id="voice_box")
                    voice_fill = gr.Button("Fill the calculator from voice")
                    voice_msg = gr.Markdown()
                with gr.Row():
                    in_age = gr.Number(label="Age (years)")
                    in_bmi = gr.Number(label="BMI")
                    in_bp = gr.Number(label="Diastolic blood pressure (mmHg)")
                    in_glu = gr.Number(label="Fasting glucose (mg/dL)")
                calc_btn = gr.Button("🔍 Check my risk", variant="primary")
                result_html = gr.HTML()
                speech_audio = gr.Audio(label="🔊 Spoken result (plays automatically, press ▶ to hear it again)",
                                        type="filepath", autoplay=True, interactive=False)
                speech_note = gr.Markdown()
                with gr.Accordion("🧠 How the AI works & how well it does", open=False):
                    gr.Markdown(model_report_md())

            with gr.Tab("💊 Medicine & Hobbies") as t_med:
                gr.Markdown("### Medicines you take")
                med_name = gr.Textbox(label="Medicine name (don't know it? use a photo below)")
                med_dose = gr.Textbox(label="Dose / note (optional)")
                with gr.Accordion("📷 Don't know the name? Upload or take a photo of the label", open=False):
                    med_img = gr.Image(label="Medicine label photo", type="pil", sources=["upload", "webcam"])
                    med_read = gr.Button("Read the name from the photo")
                    med_read_msg = gr.Markdown()
                med_save = gr.Button("Save medicine", variant="primary")
                med_msg = gr.Markdown()
                med_table = gr.Dataframe(value=med_df(None), interactive=False)
                gr.Markdown("---\n### Habits & hobbies")
                with gr.Row():
                    hb_act = gr.Slider(0, 7, value=3, step=1, label="Active days per week")
                    hb_sleep = gr.Slider(3, 12, value=7, step=0.5, label="Sleep hours per night")
                    hb_sug = gr.Slider(0, 10, value=1, step=1, label="Sugary drinks per day")
                hb_smoke = gr.Radio(["Never", "Former", "Current"], value="Never", label="Smoking")
                hb_hobby = gr.Textbox(label="Hobbies (football, walking, cooking...)")
                hb_save = gr.Button("Save habits", variant="primary")
                hb_msg = gr.Markdown()
                hb_table = gr.Dataframe(value=habit_df(None), interactive=False)

            with gr.Tab("👤 Profile") as t_prof:
                prof_md = gr.Markdown()
                prof_share = gr.Checkbox(label="Share my records with Health Extension Workers")
                prof_save = gr.Button("Save sharing choice")
                prof_msg = gr.Markdown()

            with gr.Tab("📁 My Data") as t_data:
                gr.Markdown("### Your screening history (your dataset)")
                hist_table = gr.Dataframe(value=history_df(None), interactive=False)
                data_refresh = gr.Button("Refresh")
                with gr.Row():
                    csv_btn = gr.Button("⬇ Prepare CSV")
                    pdf_btn = gr.Button("⬇ Prepare PDF")
                with gr.Row():
                    csv_file = gr.File(label="CSV file")
                    pdf_file = gr.File(label="PDF file")

            with gr.Tab("💬 HEW Messages") as t_msg:
                gr.Markdown("### 📢 Updates from HEW")
                upd_md = gr.Markdown()
                gr.Markdown("### 💬 Your conversation with HEW")
                thread_md = gr.Markdown()
                cm_type = gr.Dropdown(["Question for HEW", "Answer to a HEW question"],
                                      value="Question for HEW", label="Message type")
                cm_text = gr.Textbox(label="Your message", lines=3)
                with gr.Row():
                    cm_send = gr.Button("Send", variant="primary")
                    cm_refresh = gr.Button("Refresh")
                cm_msg = gr.Markdown()

            with gr.Tab("📍 Nearby Care") as t_care:
                gr.Markdown("### Find health facilities near you")
                gps_btn = gr.Button("📡 Use my GPS location")
                with gr.Row():
                    lat_in = gr.Textbox(label="Latitude")
                    lon_in = gr.Textbox(label="Longitude")
                with gr.Row():
                    rad_in = gr.Slider(1, 30, value=8, step=1, label="Search radius (km)")
                    typ_in = gr.Dropdown(["All", "Hospitals", "Clinics / health centres"], value="All",
                                         label="Facility type")
                find_btn = gr.Button("🔎 Recommend facilities", variant="primary")
                map_out = gr.HTML()
                fac_out = gr.HTML()

    with gr.Column(visible=False) as v_hewapp:
        with gr.Row():
            hew_welcome = gr.Markdown("## 🩺 HEW portal")
            hew_signout = gr.Button("🚪 Sign out", scale=0)
        with gr.Tabs():
            with gr.Tab("📊 Community records"):
                gr.Markdown("Only clients who agreed to share are listed.")
                with gr.Row():
                    hr_country = gr.Dropdown(["All countries"], value="All countries", label="Country")
                    hr_sort = gr.Dropdown(["Highest risk first", "Newest screening first", "Country A-Z"],
                                          value="Highest risk first", label="Sort")
                hr_btn = gr.Button("Load / refresh", variant="primary")
                hr_table = gr.Dataframe(interactive=False)
            with gr.Tab("💬 Clients: ask & advise"):
                hc_client = gr.Dropdown([], label="Client")
                hc_load = gr.Button("Open conversation")
                hc_thread = gr.Markdown()
                hc_type = gr.Dropdown(["Question for client", "Reply", "Medical suggestion"],
                                      value="Question for client", label="Message type")
                hc_text = gr.Textbox(label="Your message / specific questions / medical suggestion", lines=4)
                hc_send = gr.Button("Send to client", variant="primary")
                hc_msg = gr.Markdown()
            with gr.Tab("📢 Post update"):
                up_title = gr.Textbox(label="Title")
                up_text = gr.Textbox(label="Message to all clients", lines=4)
                up_btn = gr.Button("Publish", variant="primary")
                up_msg = gr.Markdown()
                up_list = gr.Markdown(render_updates())
            with gr.Tab("🧠 Local dataset"):
                gr.Markdown("Upload a CSV with columns **Age, BMI, BloodPressure, Glucose, Outcome** (Outcome 0/1) "
                            "to train the AI on a country's own patterns and reduce bias.")
                ds_country = gr.Dropdown(COUNTRY_NAMES, value="Ethiopia", label="Country")
                ds_file = gr.File(label="CSV file", file_types=[".csv"], type="filepath")
                ds_btn = gr.Button("Add data & retrain", variant="primary")
                ds_msg = gr.Markdown()
                ds_list = gr.Markdown(local_models_md())

    views = [v_landing, v_signup, v_login, v_forgot, v_hewlogin, v_app, v_hewapp]
    CLIENT_OUT = [welcome, prof_md, prof_share, in_age, hist_table, thread_md, upd_md, med_table, hb_table]

    b_client.click(lambda: go("login"), None, views)
    b_hew.click(lambda: go("hewlogin"), None, views)
    su_back.click(lambda: go("login"), None, views)
    lg_back.click(lambda: go("landing"), None, views)
    hw_back.click(lambda: go("landing"), None, views)
    lg_forgot.click(lambda: go("forgot"), None, views)
    fg_back.click(lambda: go("login"), None, views)

    su_pw_show.change(lambda s: (toggle_pw(s), toggle_pw(s)), su_pw_show, [su_pw, su_pw2])
    lg_show.change(toggle_pw, lg_show, lg_pw)
    fg_show.change(lambda s: (toggle_pw(s), toggle_pw(s)), fg_show, [fg_pw, fg_pw2])
    hw_show.change(toggle_pw, hw_show, hw_code)

    su_email.blur(check_email_msg, su_email, su_email_status)
    su_country.change(phone_hint, su_country, su_phone_hint)
    g_country.change(phone_hint, g_country, g_phone_hint)
    for _c in (su_day, su_month, su_year):
        _c.change(guardian_visibility, [su_day, su_month, su_year], su_guardian)

    def do_register(*a):
        msg, uid = register(*a)
        if not uid:
            return msg, "", gr.update(), gr.update(visible=False), gr.update()
        return msg, uid, uid, gr.update(visible=True), gr.update(visible=False)

    su_submit.click(do_register,
                    [su_first, su_last, su_email, su_country, su_phone, su_day, su_month, su_year,
                     g_name, g_country, g_phone, g_email, su_pw, su_pw2, su_agree, su_share],
                    [su_msg, new_account_id, su_id_box, su_done, su_submit])

    su_copy.click(None, su_id_box, None,
                  js="(v) => { try { navigator.clipboard.writeText(v); alert('ID copied: ' + v); } "
                     "catch (e) { alert('Please select and copy this ID: ' + v); } }")

    def open_account(uid, confirmed, cur):
        if not uid or uid not in DB["users"]:
            return [cur, "❌ Create your account first."] + [gr.update()] * len(views)
        if not confirmed:
            return [cur, "⚠️ Please copy your ID and tick the box before continuing."] + [gr.update()] * len(views)
        return [{"role": "client", "id": uid}, ""] + go("app")

    su_open.click(open_account, [new_account_id, su_confirm, session], [session, su_open_msg] + views) \
        .then(client_refresh, session, CLIENT_OUT)

    def to_signup():
        return go("signup") + [gr.update(visible=False), gr.update(visible=True), False, "", "", ""]

    lg_signup.click(to_signup, None, views + [su_done, su_submit, su_confirm, su_open_msg, su_msg, new_account_id])

    def do_login(uid, pw, cur):
        sess, msg = client_login(uid, pw)
        if not sess:
            return [cur, msg] + [gr.update()] * len(views)
        return [sess, ""] + go("app")

    lg_btn.click(do_login, [lg_id, lg_pw, session], [session, lg_msg] + views).then(client_refresh, session, CLIENT_OUT)
    lg_pw.submit(do_login, [lg_id, lg_pw, session], [session, lg_msg] + views).then(client_refresh, session, CLIENT_OUT)

    def do_signout():
        return [{}, "", "", ""] + go("landing")

    signout.click(do_signout, None, [session, lg_id, lg_pw, result_html] + views)
    hew_signout.click(lambda: [{}, "", ""] + go("landing"), None, [session, hw_inst, hw_code] + views)

    fg_get.click(start_reset, [fg_id, fg_email], fg_msg1)
    fg_set.click(finish_reset, [fg_id, fg_code, fg_pw, fg_pw2], fg_msg2)

    def do_hew_login(inst, code, agree, cur):
        sess, msg = hew_login(inst, code, agree)
        if not sess:
            return [cur, msg, gr.update()] + [gr.update()] * len(views)
        return [sess, "", f"## 🩺 HEW portal - {sess['inst']}"] + go("hewapp")

    hw_btn.click(do_hew_login, [hw_inst, hw_code, hw_agree, session], [session, hw_msg, hew_welcome] + views) \
        .then(hew_records, [hr_country, hr_sort], [hr_table, hr_country]) \
        .then(hew_client_choices, None, hc_client)

    lang_dd.change(None, lang_dd, None, js=LANG_JS)
    lang_dd.change(lambda c: LANG_TO_VOICE.get(c, gr.update()), lang_dd, voice_lang)

    wz_btn.click(wizard_fill, [wz_h, wz_w, wz_sym, unk_glu], [in_bmi, wz_msg])
    mic_btn.click(None, [voice_lang], None, js=VOICE_JS)
    voice_fill.click(parse_voice, voice_box, [in_age, in_bmi, in_bp, in_glu, voice_msg])
    calc_btn.click(assess, [in_age, in_bmi, in_bp, in_glu, unk_bp, unk_glu, session, lang_dd],
                   [result_html, in_bp, in_glu, speech_audio, speech_note])

    med_read.click(read_medicine_from_photo, med_img, [med_name, med_read_msg])
    _from_photo = gr.State(False)
    med_read.click(lambda: True, None, _from_photo)
    med_name.input(lambda: False, None, _from_photo)
    med_save.click(add_medicine, [med_name, med_dose, _from_photo, session], [med_msg, med_table])
    hb_save.click(add_habits, [hb_act, hb_sleep, hb_sug, hb_smoke, hb_hobby, session], [hb_msg, hb_table])

    prof_save.click(save_consent, [session, prof_share], prof_msg)
    data_refresh.click(lambda s: history_df(get_user(s)), session, hist_table)
    t_data.select(lambda s: history_df(get_user(s)), session, hist_table)
    csv_btn.click(make_csv, session, csv_file)
    pdf_btn.click(make_pdf, session, pdf_file)

    cm_send.click(send_client_msg, [cm_type, cm_text, session], [cm_msg, thread_md]) \
        .then(lambda: "", None, cm_text)
    cm_refresh.click(refresh_messages, session, [thread_md, upd_md])
    t_msg.select(refresh_messages, session, [thread_md, upd_md])

    gps_btn.click(None, None, [lat_in, lon_in], js=GPS_JS)
    find_btn.click(find_facilities, [lat_in, lon_in, rad_in, typ_in, session], [map_out, fac_out])

    hr_btn.click(hew_records, [hr_country, hr_sort], [hr_table, hr_country]) \
        .then(hew_client_choices, None, hc_client)
    hc_load.click(hew_load_thread, hc_client, hc_thread)
    hc_send.click(hew_send, [session, hc_client, hc_type, hc_text], [hc_msg, hc_thread]) \
        .then(lambda: "", None, hc_text)
    up_btn.click(hew_post_update, [session, up_title, up_text], [up_msg, up_list])
    ds_btn.click(hew_upload_dataset, [session, ds_country, ds_file], ds_msg) \
        .then(local_models_md, None, ds_list)

demo.queue().launch(share=True, debug=True, **_LAUNCH_KW)


==================== COPY EVERYTHING BETWEEN THE LINES INTO THE README (section 'Results') ====================
**Data used for this run:** Pima Indians Diabetes dataset (real public data). Trained on 460 rows, 154 rows used to choose the alert thresholds, tested once on 154 unseen rows (54 real cases).

| Measure | Baseline (always "no diabetes") | Our model |
|---|---|---|
| Accuracy | 65% | 62% |
| Recall (real cases caught) | 0% | 91% |
| Precision (alerts that were right) | n/a | 48% |
| **Missed cases (false negatives)** | all 54 real cases | 5 of 54 |
| **False alarms (false positives)** | 0 | 54 |

**By age group (test set):** Age 30 or younger: 89 people, recall 89%, missed 2, false alarms 37; Age 31 to 45: 40 people, recall 88%, missed 3, false alarms 8; Age over 45: 25 people, recall 100%, missed 0, false alarms 9.

**Example the model missed:** a person aged 34 with BMI 27.4, diastolic blood pressure 64 and glucose 114 really had diabetes, but the model gave only 27% risk,

/tmp/ipykernel_728/2917766701.py:1229: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="SugarSpot", **_BLOCKS_KW) as demo:
/tmp/ipykernel_728/2917766701.py:1229: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="SugarSpot", **_BLOCKS_KW) as demo:
/tmp/ipykernel_728/2917766701.py:1229: DeprecationWarning: The 'head' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'head' to Blocks.launch() instead.
  with gr.Blocks(title="SugarSpot", **_BLOCKS_KW) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://56f7ca0ffd03dba0f9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
